# CENG468 Veri Madenciliği · Hafta 10
## İçerik tabanlı öneri: "bu filmi beğendin, şuna benziyor"
Öğr. Gör. Dr. Duygu TOPALOĞLU · 2026-2027 Güz

---
### Bu derste nasıl çalışacağız?
İşaretler önceki defterlerle aynı:

| İşaret | Ne yapıyorsun? |
|---|---|
| 🔮 **Tahmin et** | Çalıştırmadan önce çıktıyı tahmin et ve yaz. |
| 🔧 **Değiştir** | Çalışan kodu küçük bir farkla uyarla. |
| ✍️ **Kendin yaz** | Boş hücreye kodu kendin yaz. |
| 🐞 **Hata avı** | Hata mesajını oku, nedenini bul, düzelt. |
| 🔓 **Çözüm** | Önce dene, sonra aç. |

**Bugünün sorusu:** Dün kataloğa giren, kimsenin izlemediği bir filmi kim önerebilir? Çok satanlar listesi (W5) ve sepet kuralları (W6) bunu yapamaz; ikisi de insanların davranışına bakar. Elimizde kalan tek bilgi **filmin kendisi**: türleri, etiketleri, açıklaması. Bu defterde sunumdaki adımları aynı sırayla kendimiz yazıyoruz: içeriği tabloya çevir → benzerliği önce sezgiyle sonra kosinüsle ölç → filme öneri → kullanıcıya öneri → metinden vektöre (TF-IDF).

**Veri:** MovieLens ml-latest-small (`Veri/movielens/BENIOKU.md`). Bölüm 5 için ayrıca Kaggle "The Movies Dataset" (`Veri/movies_metadata/BENIOKU.md`); bu bölüm derste zaman kalırsa, kalmazsa ev alıştırması.

---
## Bölüm 0: Veriyi tanı (10 dk)
`movies.csv` üç sütun: film numarası, ad, türler. Türler tek hücrede `|` ile ayrılmış. `ratings.csv` kullanıcıların filmlere verdiği puanlar (0,5–5).

In [ ]:
import os
import numpy as np
import pandas as pd

VERI = os.path.join("Veri", "movielens", "ml-latest-small")
if not os.path.exists(os.path.join(VERI, "movies.csv")):           # zip bir kat fazla açıldıysa
    VERI = os.path.join(VERI, "ml-latest-small")

filmler = pd.read_csv(os.path.join(VERI, "movies.csv"))
puanlar = pd.read_csv(os.path.join(VERI, "ratings.csv"))
print(filmler.shape, puanlar.shape)

# Veri kirliliği 1: birkaç film adı iki kez geçiyor (farklı movieId, aynı ad).
# Adı anahtar olarak kullanacağımız için tekrarları atıyoruz.
tekrar = filmler["title"].duplicated(keep=False)
print("Tekrar eden adlar:", filmler.loc[tekrar, "title"].unique())
filmler = filmler.drop_duplicates("title").reset_index(drop=True)
filmler.head()

**🔮 Tahmin et:** Sunumun başındaki soru: hiç puan almamış kaç film var? Yani `movies.csv`'de olup `ratings.csv`'de hiç geçmeyen film sayısı. 0 mı, 10 civarı mı, 100'den fazla mı?

*Tahminim:* ...

In [ ]:
puansiz = ~filmler["movieId"].isin(puanlar["movieId"])
print("Hiç puan almamış film:", puansiz.sum())
filmler.loc[puansiz, ["title", "genres"]].head(8)

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e8f5e9; color:#1a1a1a; border-left:6px solid #2e7d32; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Yanıt</b>


**18 film** hiç puan almamış. Çok satanlar listesi (izlenme sayısına göre) ve sepet kuralları (birlikte beğenilme) bu 18 filmi hiçbir zaman öneremez. Türleri belli olduğu için içerik tabanlı yöntem önerebilir. Bu, yöntemin varlık nedeni: **yeni öğe** sorunu (new item / first-rater problem). Yeni **kullanıcı** sorunu ise bu yöntemde de çözülmez; Bölüm 3'te göreceğiz.


</div>

</details>

**🔮 Tahmin et:** `filmler["genres"]` sütununda kaç **farklı tür** var? (Toy Story'nin satırında beş tür yazıyor; ama farklı tür sayısı kaç?) Nasıl sayarsın?

*Tahminim:* ...

In [ ]:
turler = filmler["genres"].str.get_dummies("|")   # her tür bir sütun, 1 = film bu türde
print(turler.shape)
turler.columns.tolist()

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e8f5e9; color:#1a1a1a; border-left:6px solid #2e7d32; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Kod ne yapıyor?</b>


`str.get_dummies("|")`: hücredeki metni `|` işaretinden bölüp her parça için bir sütun açar; film o türdeyse 1, değilse 0. Sonuç 9.737 film × 20 sütun (ham dosyada 9.742 film; 5 tekrarlı ad atıldı). Sütunlardan biri `(no genres listed)`: türü yazılmamış 34 film. Bu da bir veri kirliliği; içerik tabanlı yöntem, içerik yoksa çalışmaz (sepet analizinde böyle bir sorun yoktu).

Bu tabloya **öğe profili** (item profile) diyoruz: her film, 20 sayıdan oluşan bir **vektör**. Sunumdaki 7. slayt.

Yaygın hata: `pd.get_dummies(filmler["genres"])` (başında `.str` olmadan) hata vermez ama `|` işaretini bölmez; 951 sütun üretir, her tür **bileşimi** ayrı sütun olur. Sessiz yanlış; `shape` ile her zaman kontrol edin.


</div>

</details>

---
## Bölüm 1: Benzerlik elle (20 dk)
Bütün tabloya geçmeden sunumdaki beş filmle çalışalım.

In [ ]:
secilen = [1, 2, 6, 10, 7]                   # Toy Story, Jumanji, Heat, GoldenEye, Sabrina
mini = turler.loc[filmler["movieId"].isin(secilen)]
mini.index = filmler.loc[mini.index, "title"].str.replace(r" \(\d{4}\)", "", regex=True)
mini = mini.loc[:, mini.sum() > 0]             # bu beş filmde hiç geçmeyen türleri gizle
mini

### 1.1 Sezgiyle: ortak türleri say
**🔮 Tahmin et:** Toy Story ile Jumanji'nin kaç **ortak** türü var? Toy Story ile GoldenEye'ın? Tablodan say.

*Tahminim:* ...

Kodla: iki 0/1 satırını çarpıp toplarsan ortak 1'leri saymış olursun.

In [ ]:
ortak = (mini * mini.loc["Toy Story"]).sum(axis=1)     # her film × Toy Story satırı, satır toplamı
ortak.sort_values(ascending=False)

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e8f5e9; color:#1a1a1a; border-left:6px solid #2e7d32; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Yanıt</b>


Toy Story ile Jumanji'nin **3 ortak türü** var (Adventure, Children, Fantasy); GoldenEye ile **1** (Adventure); Sabrina ile 1 (Comedy); Heat ile 0. `mini * mini.loc["Toy Story"]`: her satır Toy Story satırıyla hücre hücre çarpılır, yalnız ikisinde de 1 olan hücreler 1 kalır; `sum(axis=1)` satır toplamı. Toy Story'nin kendisiyle ortak türü 5 (tür sayısı).


</div>

</details>

**🔧 Değiştir:** Sunumdaki kurgu film **"Her Şey"**: dokuz türün dokuzu işaretli. Tabloya ekle ve ortak tür sayılarını yeniden hesapla. "Her Şey" Toy Story'ye Jumanji'den daha mı benzer çıkıyor?

In [ ]:
mini2 = mini.copy()
mini2.loc["Her Şey"] = 1
# Kodunu buraya yaz: ortak tür sayılarını mini2 için hesapla

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#ffebee; color:#1a1a1a; border-left:6px solid #c62828; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Yanıt</b>


```python
ortak2 = (mini2 * mini2.loc["Toy Story"]).sum(axis=1)
```
"Her Şey" ile ortak tür sayısı **5**: Jumanji'nin 3'ünü geçti. Oysa aynı film Heat'e de (3 ortak) GoldenEye'a da (3) "en benzer" çıkar. Ortak tür saymak çok türlü filmi kayırır; sayıyı filmlerin kaç türü olduğuna göre ölçeklemek gerekir. Kosinüs tam bunu yapar.


</div>

</details>

### 1.2 Kosinüs benzerliği (cosine similarity)
$$\cos(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|} = \frac{\sum_i a_i b_i}{\sqrt{\sum_i a_i^2}\,\sqrt{\sum_i b_i^2}}$$

0/1 vektörlerde pay = **ortak tür sayısı** (bir önceki hücrede hesapladığın şey), paydadaki her kök = **o filmin tür sayısının karekökü**. Toy Story (5 tür) ile Jumanji (3 tür), ortak 3: $3 / (\sqrt5 \cdot \sqrt3) = 0{,}775$.

**✍️ Kendin yaz:** `kosinus(a, b)` fonksiyonu. `a` ve `b` birer satır (`mini.loc["Toy Story"]` gibi). İpucu: `(a * b).sum()`, `np.sqrt`.

In [ ]:
def kosinus(a, b):
    ...

print("Toy Story – Jumanji  :", round(kosinus(mini2.loc["Toy Story"], mini2.loc["Jumanji"]), 3))
print("Toy Story – Her Şey  :", round(kosinus(mini2.loc["Toy Story"], mini2.loc["Her Şey"]), 3))
print("Toy Story – GoldenEye:", round(kosinus(mini2.loc["Toy Story"], mini2.loc["GoldenEye"]), 3))
print("Heat – GoldenEye     :", round(kosinus(mini2.loc["Heat"], mini2.loc["GoldenEye"]), 3))

<details><summary>💡 İpucu 1</summary>

Pay: `(a * b).sum()`. Payda: `np.sqrt((a ** 2).sum()) * np.sqrt((b ** 2).sum())`. 0/1 vektörde `a ** 2` ile `a` aynı şey; formülü genel yazmak metin vektörlerinde de işe yarar.

</details>

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e3f2fd; color:#1a1a1a; border-left:6px solid #1565c0; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Çözüm</b>


```python
def kosinus(a, b):
    return (a * b).sum() / (np.sqrt((a ** 2).sum()) * np.sqrt((b ** 2).sum()))
```
Toy Story – Jumanji **0,775** · Toy Story – Her Şey **0,745** · Toy Story – GoldenEye 0,258 · Heat – GoldenEye 0,667. "Her Şey" artık Jumanji'nin gerisinde: $5 / (\sqrt5 \cdot \sqrt9) = 0{,}745$. Sunumdaki 9. slaytla karşılaştır.


</div>

</details>

**🔮 Düşün:** Benzerliği **Öklid uzaklığı** ile de ölçebilirdik: $\sqrt{\sum_i (a_i - b_i)^2}$. 0/1 vektörlerde bu, **yalnız birinde olan** türlerin sayısının karekökü. Toy Story – Jumanji için kaç? Toy Story'nin her türü üç kez etiketlenmiş olsaydı (satır 3'le çarpılsaydı) Öklid ve kosinüs ne derdi?

*Cevabım:* ...

In [ ]:
def oklid(a, b):
    return np.sqrt(((a - b) ** 2).sum())

for x, y in [("Toy Story", "Jumanji"), ("Toy Story", "GoldenEye"), ("Heat", "GoldenEye")]:
    print(f"{x} – {y}: kosinüs {kosinus(mini.loc[x], mini.loc[y]):.3f}  Öklid {oklid(mini.loc[x], mini.loc[y]):.3f}")

uc_kat = mini.loc["Toy Story"] * 3
print(f"Toy Story – Toy Story×3: kosinüs {kosinus(mini.loc['Toy Story'], uc_kat):.3f}  Öklid {oklid(mini.loc['Toy Story'], uc_kat):.3f}")

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#ffebee; color:#1a1a1a; border-left:6px solid #c62828; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Neden kosinüs?</b>


Öklid'de Toy Story – Jumanji $\sqrt2 = 1{,}41$ (Animation ve Comedy yalnız Toy Story'de), Toy Story – GoldenEye $\sqrt6 = 2{,}45$. Toy Story ile üç kat etiketlenmiş kopyası: kosinüs **1,0**, Öklid **4,47**. Öklid'e göre film kendi kopyasına Jumanji'den (1,41) üç kat uzak. Kosinüs vektörün **uzunluğuna** değil **yönüne** bakar; bu yüzden tür ve metin gibi "kaç tane" sorusunun önemsiz olduğu verilerde kosinüs kullanılır. Sunumdaki 10. slayt: aynı doğrultuda iki ok, uzunlukları farklı; Öklid uzak der, kosinüs aynı der.


</div>

</details>

---
## Bölüm 2: Filme öneri, bütün filmler (15 dk)
Beş filmde elle yaptığımızı 9.737 filmde kütüphaneyle yapalım. `cosine_similarity` her film çiftinin kosinüsünü tek seferde hesaplar.

**🔮 Tahmin et:** 9.737 film için benzerlik matrisi kaç satır × kaç sütun olur? Kaç sayı? (Her sayı 8 bayt; kaç MB?)

*Tahminim:* ...

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

n_film = len(filmler)
print(f"Tam matris: {n_film} × {n_film} = {n_film**2:,} sayı × 8 bayt = {n_film**2 * 8 / 1e6:,.0f} MB")

# O kadar belleği harcamayalım: ilk 5 film için deneyelim
ornek = cosine_similarity(turler.values[:5])
pd.DataFrame(ornek, index=filmler["title"][:5], columns=filmler["title"][:5]).round(2)

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e8f5e9; color:#1a1a1a; border-left:6px solid #2e7d32; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Kod ne yapıyor?</b>


`cosine_similarity(X)` X'in her satır çifti için kosinüsü hesaplar; Bölüm 1'deki fonksiyonun hızlı sürümü. Köşegen 1,0 (film kendine benzer). Tam matris 758 MB; 100.000 filmlik bir katalogda 80 GB olurdu (n² büyür). Bu yüzden aşağıda **tek film için** anlık hesaplayacağız: `cosine_similarity(bir_satir, bütün_tablo)` → 1 × 9.737, 78 KB.


</div>

</details>

**✍️ Kendin yaz:** `benzer_filmler(ad, n=10)`: verilen filmin tür satırını bütün tabloyla karşılaştır, kosinüsü en yüksek `n` filmi döndür (filmin kendisi hariç). İskelet hazır; üç boşluğu doldur.

In [ ]:
def benzer_filmler(ad, n=10):
    i = filmler.index[filmler["title"] == ad][0]                       # filmin satır numarası
    skor = cosine_similarity(..., ...)[0]                              # 1 × 9.737
    skor = pd.Series(skor, index=filmler["title"])
    return skor.drop(...).nlargest(n)

benzer_filmler("Toy Story (1995)")

<details><summary>💡 İpucu 1</summary>

İlk boşluklar: filmin satırı `turler.values[[i]]` (çift köşeli parantez: 1 × 20 tablo olsun) ve bütün tablo `turler.values`.

</details>

<details><summary>💡 İpucu 2</summary>

Son boşluk: filmin kendisini düşürmek için `skor.drop(ad)`.

</details>

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e3f2fd; color:#1a1a1a; border-left:6px solid #1565c0; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Çözüm</b>


```python
def benzer_filmler(ad, n=10):
    i = filmler.index[filmler["title"] == ad][0]
    skor = cosine_similarity(turler.values[[i]], turler.values)[0]
    skor = pd.Series(skor, index=filmler["title"])
    return skor.drop(ad).nlargest(n)
```
`turler.values[i]` yazarsan tek boyutlu dizi gelir ve `cosine_similarity` "Expected 2D array, got 1D array instead" hatası verir; `[[i]]` ile 1 × 20'lik tablo alırsın.


</div>

</details>

**🔮 Düşün:** Listedeki filmlerin hepsinin kosinüsü 1,0. Neden? Hangisi "en" benzer; veri bunu söyleyebilir mi? Bu liste "Toy Story'yi beğenene öner" diye gösterilse ne olur?

*Cevabım:* ...

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#ffebee; color:#1a1a1a; border-left:6px solid #c62828; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Aşırı özelleşme (over-specialization)</b>


Kosinüs 1,0 = **tamamen aynı tür kümesi**. Toy Story'nin beş türünü birebir taşıyan 12 film var; yöntem aralarında ayrım yapamaz, sıra `nlargest`'in rastladığı sıra. Yalnız türle bu kadar ayrım yapılabiliyor (sınırlı içerik çözümlemesi). Üstelik hepsi çocuk animasyonu: kullanıcı başka hiçbir şey görmez. İçerik tabanlı yöntemin bilinen zayıflığı: **beğendiğinin aynısını önerir, şaşırtmaz.** Sepet analizi Kill Bill 2'den Kill Bill 1'e gitti ama tür bilgisi kullanmadan, insanların davranışından. W5'teki Müşteri 12417 listesi de aynı şeydi: "aynı şeyden bir tane daha".

İki iyileştirme: beraberlik bozucu olarak **popülerlik** (hemen aşağıda) ve türlere ek **etiketler** ile daha zengin öğe profili (Bölüm 4).


</div>

</details>

**🔧 Değiştir:** Beraberlikleri bozmak için filmin aldığı puan sayısını kullan: kosinüsü eşit olanlar arasında çok puanlanan öne geçsin. Aşağıdaki hücre bunu yapıyor; sonra `"Heat (1995)"` için de çalıştır.

In [ ]:
puan_sayisi = puanlar.merge(filmler[["movieId", "title"]]).groupby("title").size()   # film başına puan sayısı

def benzer_filmler2(ad, n=10):
    aday = benzer_filmler(ad, n=len(filmler) - 1)      # bütün filmler, kosinüse göre sıralı
    tablo = pd.DataFrame({"kosinus": aday, "puan_sayisi": puan_sayisi.reindex(aday.index).fillna(0).astype(int)})
    return tablo.sort_values(["kosinus", "puan_sayisi"], ascending=False).head(n)

benzer_filmler2("Toy Story (1995)")

In [ ]:
# Kodunu buraya yaz: Heat (1995)

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#ffebee; color:#1a1a1a; border-left:6px solid #c62828; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Yorum</b>


`Heat (1995)` için de ilk 10 filmin kosinüsü 1,0 (Heat: Action, Crime, Thriller; aynı üç türü taşıyan çok film var). Puan sayısına göre sıralayınca Batman (1989), Die Hard, Die Hard: With a Vengeance, Kill Bill: Vol. 1 öne çıkar. Popülerlik yalnız **beraberliği bozar**; filmi türce daha benzer yapmaz. Bağ kırma (tie-break) kuralı bir **karar**dır: yıl, ortalama puan ya da rastgele de olabilirdi; projende seçimini gerekçelendir.


</div>

</details>

---
## Bölüm 3: Kullanıcıya öneri, kullanıcı profili (15 dk)
Şimdiye kadar "bu filme benzeyen" sorusunu sorduk. Asıl soru "**bu kullanıcıya** ne önerelim?" Sunumdaki 14. slayt: kullanıcının beğendiği filmlerin tür vektörlerinin **ortalaması** = **kullanıcı profili** ("hayalî film"). Sonra profil ile izlemediği filmler arasında kosinüs.

In [ ]:
KULLANICI = 1
izledi = puanlar[puanlar["userId"] == KULLANICI]
begendi = izledi[izledi["rating"] >= 4]
begendi_idx = filmler.index[filmler["movieId"].isin(begendi["movieId"])]

profil = turler.loc[begendi_idx].mean()          # her tür için: beğendiklerinin yüzde kaçı bu türde?
print(f"Kullanıcı {KULLANICI}: {len(izledi)} film izlemiş, {len(begendi)} filmi beğenmiş (puan ≥ 4)")
profil.sort_values(ascending=False).head(8).round(2)

**🔮 Düşün:** Profil 20 sayıdan oluşan bir vektör; Toy Story'nin vektörüyle aynı boyda. Profilde "Action 0,38" ne demek? Profil bir "film" gibi davranabilir mi?

*Cevabım:* ...

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e3f2fd; color:#1a1a1a; border-left:6px solid #1565c0; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Yanıt</b>


Profil, kullanıcının beğendiği filmlerin tür vektörlerinin **ortalaması**; film vektörleriyle aynı uzayda. "Action 0,38": beğendiği 200 filmin %38'i Action. Profil gerçek bir film değil, "ortalama bir film"; ama aynı boyda olduğu için kosinüsle bütün filmlerle karşılaştırılabilir. İşte bu yüzden profil bir film gibi davranabilir. Sunumdaki 6 filmlik örnekte Comedy 0,67 · Adventure 0,67 · SuperHero 1,00 · SciFi 0,33 de aynı hesap.


</div>

</details>

**✍️ Kendin yaz:** `kullaniciya_oner(profil, kullanici, n=10)`: profil ile **bütün** filmler arasında kosinüs hesapla (`cosine_similarity(profil.values.reshape(1, -1), turler.values)`), kullanıcının **zaten puanladığı** filmleri çıkar (`kullanici`: kimin puanlarına bakılacağı), en yüksek `n` filmi döndür. Puan sayısını da yanına koy (Bölüm 2'deki `puan_sayisi`).

In [ ]:
def kullaniciya_oner(profil, kullanici, n=10):
    ...

kullaniciya_oner(profil, KULLANICI)

<details><summary>💡 İpucu 1</summary>

Bölüm 2'deki `benzer_filmler` ile aynı iskelet; fark: satır yerine `profil`, düşürülecekler filmin kendisi yerine kullanıcının izledikleri.

</details>

<details><summary>💡 İpucu 2</summary>

İzlenenlerin adı: `filmler.loc[filmler['movieId'].isin(puanlar.loc[puanlar['userId'] == kullanici, 'movieId']), 'title']`.

</details>

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e3f2fd; color:#1a1a1a; border-left:6px solid #1565c0; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Çözüm ve yorum</b>


```python
def kullaniciya_oner(profil, kullanici, n=10):
    skor = cosine_similarity(profil.values.reshape(1, -1), turler.values)[0]
    skor = pd.Series(skor, index=filmler["title"])
    izlenen = filmler.loc[filmler["movieId"].isin(puanlar.loc[puanlar["userId"] == kullanici, "movieId"]), "title"]
    skor = skor.drop(izlenen)
    tablo = pd.DataFrame({"skor": skor, "puan_sayisi": puan_sayisi.reindex(skor.index).fillna(0).astype(int)})
    tablo["turler"] = filmler.set_index("title").loc[tablo.index, "genres"]
    return tablo.sort_values(["skor", "puan_sayisi"], ascending=False).head(n)
```
Sunumdaki 15. slayt: Dragonheart 2, Hunting Party, Flashback… Hepsi Action·Adventure·Comedy·Drama karışımı; ilk sekizi en çok 3 puan almış, kimsenin bilmediği filmler. Profil "her şeyden biraz" olduğu için ona en yakın düşen filmler çok türlü, sıradan filmler. Skor yalnız türlerden geliyor; filmin iyi ya da kötü olduğu formülde yok. İyi yanı: hiç puanı olmayan 18 film de böyle önerilebilir. Kötü yanı: kötü film de önerilebilir.


</div>

</details>

**🔧 Değiştir:** Beğeni eşiğini değiştir: `rating >= 3` ve `rating >= 5` ile profili yeniden kur, ilk 10'u karşılaştır. Profil nasıl değişiyor? Hangi eşik "beğendi" demek; bu kararı projende nasıl gerekçelendirirsin?

In [ ]:
# Kodunu buraya yaz

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#f3e5f5; color:#1a1a1a; border-left:6px solid #6a1b9a; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Yorum</b>


Eşik 3 ile 226 film, eşik 5 ile 124 film profile girer; Kullanıcı 1 çok cömert puan verdiği için (ortalaması 4,37) üç profil birbirine yakın, ilk 10 az değişir. Puanları daha yayılmış bir kullanıcıda fark büyür. Düşük puan verdiği filmler profile hiç girmiyor; bu da bir karar. Puanla ağırlıklandırmak ya da kullanıcı ortalamasını çıkarmak başka seçenekler (hoca notunda; sorarsanız anlatırım).


</div>

</details>

---
## Bölüm 4: Metinden vektöre, etiketler ile TF-IDF (15 dk)
Sunumdaki 11–12. slaytlar: tür yerine **metin** olsaydı her kelime bir sütun olurdu, hücre kelimenin kaç kez geçtiği; sonra her kelime nadirliğiyle ağırlıklanır (TF-IDF). ml-latest-small'da film açıklaması yok ama `tags.csv` var: kullanıcıların filmlere yazdığı kısa etiketler. Film başına etiketleri birleştirince kısa bir "metin" elde ederiz.

In [ ]:
etiketler = pd.read_csv(os.path.join(VERI, "tags.csv"))
print(etiketler.shape)
etiketler.head()

**🔮 Tahmin et:** 9.737 filmin yüzde kaçında en az bir etiket vardır? %5 mi, %15 mi, %50 mi?

*Tahminim:* ...

In [ ]:
film_etiket = etiketler.groupby("movieId")["tag"].apply(lambda s: " ".join(s.astype(str)))   # film başına tek metin
filmler["etiket"] = filmler["movieId"].map(film_etiket)
print("Etiketli film:", filmler["etiket"].notna().sum(), f"({filmler['etiket'].notna().mean() * 100:.1f}%)")
print("Toy Story'nin etiketleri:", filmler.loc[filmler["title"] == "Toy Story (1995)", "etiket"].values[0])

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e8f5e9; color:#1a1a1a; border-left:6px solid #2e7d32; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Yanıt</b>


Yalnız **1.572 film (%16)** etiketli; 8.165 filmin etiketi yok (NaN). Toy Story'nin bütün etiket metni `"pixar pixar fun"`: üç kelime, biri iki kez. Etiket, türden zengin ama seyrek bir içerik. Katalogun %84'ü için bu yol hiçbir şey söyleyemez; bu yüzden gerçek sistemler türleri ve etiketleri **birlikte** kullanır.


</div>

</details>

**🐞 Hata avı:** Aşağıdaki hücre hata veriyor. Hata mesajının **son satırını** oku: hangi değer geçersiz bulunmuş? Alttaki hücrede düzelt.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vektorlestirici = TfidfVectorizer()
X_etiket = vektorlestirici.fit_transform(filmler["etiket"])

In [ ]:
# Düzeltilmiş kod

<details><summary>💡 İpucu 1</summary>

`ValueError: np.nan is an invalid document`: etiketi olmayan filmlerin hücresi NaN. Metin değil. Boş metne çevir: `.fillna("")`.

</details>

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e3f2fd; color:#1a1a1a; border-left:6px solid #1565c0; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Çözüm</b>


```python
X_etiket = vektorlestirici.fit_transform(filmler["etiket"].fillna(""))
```
Sonuç 9.737 film × 1.744 kelime, **seyrek** matris (hücrelerin %99,97'si sıfır; bu yüzden `cosine_similarity` gibi fonksiyonlar bunu yoğun tabloya çevirmeden işler). Etiketi olmayan 8.165 film sıfır satır: hiçbir filme benzemez, kosinüsü 0. `TfidfVectorizer` kelimeleri küçük harfe çevirir, noktalamadan böler, her kelimeyi nadirliğiyle ağırlıklar ve satırları normalize eder. Formülü ders kitabındakinden biraz farklı (idf = ln((1+N)/(1+n)) + 1); fikir aynı: her filmde geçen kelime az, az filmde geçen kelime çok sayılır.


</div>

</details>

**✍️ Kendin yaz:** `benzer_filmler_etiket(ad, n=10)`: Bölüm 2'deki `benzer_filmler` ile aynı iskelet, bu kez `turler.values` yerine `X_etiket`. Seyrek matriste satır seçimi `X_etiket[i]` (tek köşeli parantez yeter). Toy Story için çalıştır.

In [ ]:
def benzer_filmler_etiket(ad, n=10):
    ...

benzer_filmler_etiket("Toy Story (1995)")

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e3f2fd; color:#1a1a1a; border-left:6px solid #1565c0; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Çözüm ve yorum</b>


```python
def benzer_filmler_etiket(ad, n=10):
    i = filmler.index[filmler["title"] == ad][0]
    skor = cosine_similarity(X_etiket[i], X_etiket)[0]
    skor = pd.Series(skor, index=filmler["title"])
    return skor.drop(ad).nlargest(n).round(2)
```
Bug's Life 0,90 · Toy Story 2 0,58 · **Guardians of the Galaxy 2 0,43** · Up 0,23 · The Lego Movie 0,15 … 9. ve 10. sıradakiler kosinüs **0,00** ile listeye giriyor: Toy Story'ye benzerliği sıfırdan büyük yalnız **8 film** var. Listeyi "en yüksek 10" diye kesmek yetmez; "benzerliği 0'dan büyük" eşiği de gerekir.

**🔮 Düşün:** Guardians of the Galaxy 2 neden üçüncü sırada? Etiketine bak (`filmler.loc[filmler["title"].str.startswith("Guardians of the Galaxy 2"), "etiket"]`). Tek etiketi "fun". TF-IDF **kelimeyi** eşler, **anlamı** değil.


</div>

</details>

---
## Bölüm 5: Açıklama metni ile aynı kod (ev alıştırması; derste zaman kalırsa 10 dk)
Etiketler seyrekti. Film **açıklaması** (overview) her filmde var ama ml-latest-small'da bulunmuyor. Kaggle "The Movies Dataset" (`movies_metadata.csv`, TMDB'den çekilmiş 45.000 film) açıklamayı veriyor; `links.csv`'deki `tmdbId` ile MovieLens filmlerine bağlanır. İndirme yönergesi: `Veri/movies_metadata/BENIOKU.md`. Eski dönemlerde içerik tabanlı öneriyi tam bu veriyle anlatıyorduk; aşağıdaki hücreler o kodun bugünkü akışa uyarlanmış hâli.

In [ ]:
META = os.path.join("Veri", "movies_metadata", "movies_metadata.csv")
if os.path.exists(META):
    meta = pd.read_csv(META, low_memory=False)
    meta["id"] = pd.to_numeric(meta["id"], errors="coerce")              # Veri kirliliği: 3 satırda id yerine tarih yazıyor
    meta = meta.dropna(subset=["id"]).drop_duplicates("id")
    meta["id"] = meta["id"].astype(int)

    baglanti = pd.read_csv(os.path.join(VERI, "links.csv"))
    filmler2 = (filmler.merge(baglanti[["movieId", "tmdbId"]], on="movieId")
                       .merge(meta[["id", "overview"]], left_on="tmdbId", right_on="id", how="left"))
    print("Açıklaması olan film:", filmler2["overview"].notna().sum(), "/", len(filmler2))
    print(filmler2.loc[filmler2["title"] == "Sherlock Holmes (2009)", "overview"].values[0])
else:
    print("movies_metadata.csv yok; Veri/movies_metadata/BENIOKU.md'deki yönergeyle indirin.")

**✍️ Kendin yaz:** Bölüm 4'teki akışın aynısı, bu kez `overview` sütununda ve `TfidfVectorizer(stop_words="english")` ile (İngilizce "the, and, of" gibi kelimeler atılır). `benzer_filmler_aciklama(ad, n=10)` fonksiyonunu yaz; "Sherlock Holmes (2009)" ve "Toy Story (1995)" için çalıştır.

In [ ]:
# Kodunu buraya yaz

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e3f2fd; color:#1a1a1a; border-left:6px solid #1565c0; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Çözüm ve yorum</b>


```python
vek2 = TfidfVectorizer(stop_words="english")
X_aciklama = vek2.fit_transform(filmler2["overview"].fillna(""))

def benzer_filmler_aciklama(ad, n=10):
    i = filmler2.index[filmler2["title"] == ad][0]
    skor = cosine_similarity(X_aciklama[i], X_aciklama)[0]
    skor = pd.Series(skor, index=filmler2["title"])
    return skor.drop(ad).nlargest(n).round(2)
```
9.737 film × 30.449 kelime. Sherlock Holmes (2009) için ilk 10'un 9'u Sherlock Holmes filmi (They Might Be Giants 0,37, Without a Clue 0,32, Young Sherlock Holmes 0,29 …): açıklamalarda "Holmes", "Watson", "detective" ortak. Toy Story için Toy Story 3 (0,53) ve Toy Story 2 (0,45), ardından **The 40-Year-Old Virgin (0,30)**: açıklamasında "toys" kelimesi geçiyor. Kelime eşleşmesi anlam eşleşmesi değil; TF-IDF'in sınırı burada da görünüyor. Yine de türle kıyaslandığında açıklama metni filmleri çok daha iyi ayırıyor: 12 film 1,00 değil, her filmin farklı bir skoru var.


</div>

</details>

---
#### ⭐ Ek alıştırmalar: erken bitirenler ve ev için
**E1.** `benzer_filmler2` ile `benzer_filmler_etiket` çıktılarını **"Heat (1995)"** için karşılaştır. Hangi listede hangi filmler var; hangisi daha "makul"? Etiketle bulunan listenin kaç filmi kosinüs 0 ile girdi?

In [ ]:
# E1

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e8f5e9; color:#1a1a1a; border-left:6px solid #2e7d32; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Yanıt</b>


Türle: Action·Crime·Thriller üçlüsünü taşıyan popüler filmler. Etiketle: Heat'in etiketleri (`filmler.loc[filmler["title"] == "Heat (1995)", "etiket"]`) hangi kelimeleri içeriyorsa onları paylaşan filmler; listenin sonunda kosinüsü 0 olan filmler var. İki liste birbirinden farklı; "doğru" liste diye bir şey yok, değerlendirme haftasında bunu ölçmeyi öğreneceğiz.


</div>

</details>

**E2.** Türler ve etiketleri **birleştir**: `scipy.sparse.hstack([X_tur_normalize, X_etiket])` ile tek matris kur (tür satırlarını önce `sklearn.preprocessing.normalize` ile birim uzunluğa getir) ve Toy Story için en benzer 10'u bul. Bug's Life ilk sırada mı?

In [ ]:
# E2

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e8f5e9; color:#1a1a1a; border-left:6px solid #2e7d32; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Yanıt</b>


Bug's Life 0,90, Toy Story 2 0,79, ardından tür ikizleri 0,71 (= tür kosinüsü 1 ile etiket kosinüsü 0'ın ortalaması). Etiketi olan filmler öne çıkıyor, etiketi olmayanlar tür benzerliğinin yarısıyla kalıyor. Birleştirme oranı (burada yarı yarıya) da bir karar.


</div>

</details>

**E3.** Kullanıcı 53'ün bütün puanları 5,0. Bölüm 3'teki profil bu kullanıcı için ne verir? (İpucu: `KULLANICI = 53` ile hücreleri yeniden çalıştır.) Ya her puanını kullanıcı ortalamasından çıkarıp ağırlık olarak kullansaydık (puan − ortalama)? Profil ne olurdu, kosinüs ne olurdu?

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#f3e5f5; color:#1a1a1a; border-left:6px solid #6a1b9a; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Yanıt</b>


Eşik 4 ile 20 filmin 20'si profile girer, profil normal bir ortalama; öneri üretilir. Ortalamadan arındırılmış ağırlıkla (puan − 5,0 = 0) bütün ağırlıklar 0, profil **sıfır vektör**, kosinüs tanımsız (0/0; NumPy `nan`, scikit-learn 0 döndürür). Gerçek sistemde bu köşe durumunu (`norm == 0`) yakalamak gerekir. Ders: her formül her kullanıcıda çalışmaz; kodu yazarken "bu kullanıcı için payda sıfır olabilir mi" diye sorun.


</div>

</details>

---
### 🧠 Kendini sına (5 soru)
Önce cevabını **yaz**, sonra alttaki hücreyi çalıştırıp kontrol et.

1. Film A = (1, 1, 0, 0), Film B = (1, 1, 1, 1). Ortak tür sayısı? Kosinüs?
2. Film C = (1, 0, 0, 0), Film D = (1, 0, 0, 0). Kosinüs? Öklid? Bu iki film aynı film mi?
3. Bir kullanıcı (1,1,0,0) ve (1,0,1,0) filmlerini beğendi. Profili?
4. Hiç puanı olmayan yeni bir filmi hangi yöntem önerebilir: çok satanlar, sepet analizi, içerik tabanlı? Hiç puan vermemiş yeni bir kullanıcıya hangisi?
5. "ve", "bir" kelimelerinin TF-IDF ağırlığı neden sıfıra yakındır?

In [ ]:
A, B, C, D = np.array([1, 1, 0, 0]), np.array([1, 1, 1, 1]), np.array([1, 0, 0, 0]), np.array([1, 0, 0, 0])
print(1, (A * B).sum(), round(kosinus(A, B), 3))
print(2, kosinus(C, D), oklid(C, D))
print(3, np.mean([[1, 1, 0, 0], [1, 0, 1, 0]], axis=0))

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e8f5e9; color:#1a1a1a; border-left:6px solid #2e7d32; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Cevaplar</b>


1. Ortak 2; kosinüs $2/(\sqrt2 \cdot \sqrt4) = 0{,}707$.
2. Kosinüs 1, Öklid 0; **aynı film değil**, aynı tür kümesi. Benzerlik ölçüsü filmi değil temsilini karşılaştırır.
3. (1, 0,5, 0,5, 0).
4. Yeni film: yalnız içerik tabanlı. Yeni kullanıcı: hiçbiri kişisel öneri yapamaz; çok satanlar listesi "herkese aynı" yedeğidir.
5. Her belgede geçerler; idf = log(N / N) = 0 (scikit-learn'de en küçük değer). Nadir kelime değerlidir.


</div>

</details>

---
## Kapanış
**Bugün:** Öğe profili (tür vektörü) → ortak tür sayma → kosinüs benzerliği (elle ve kütüphaneyle) → "bu filme benzer" → kullanıcı profili → "bu kullanıcıya" → etiket ve açıklama metninden TF-IDF ile aynı akış. Yöntem **ürünün ne olduğuna** bakıyor; sepet analizi **kimin ne aldığına** bakıyordu.

| | Sepet analizi | İçerik tabanlı |
|---|---|---|
| Girdi | Kim neyi birlikte aldı | Ürünün öznitelikleri |
| Yeni ürün | Kimse almadıysa öneremez | Öznitelikleri varsa önerir |
| Yeni kullanıcı | Sorun değil (kişisel değil) | Profil kurulamaz |
| Zayıflık | Popüler ürün her yerde | Beğendiğinin aynısı, şaşırtmaz |

**Gelecek hafta:** İşbirlikçi filtreleme. Ürüne bakmadan, "senin gibi beğenen insanlar şunu da beğendi" diyeceğiz; 4 kullanıcı × 4 film tablosunda benzerlik ağırlıklarını elle hesaplayacağız. Toy Story'yi sevene Forrest Gump'ı ancak bu yöntem önerebilir.

**Projen için soru:** Senin verinde öğelerin öznitelikleri var mı (kategori, açıklama, marka)? Yoksa içerik tabanlı yöntem senin verine uygulanamaz; bunu raporunda bir cümleyle yazabilmelisin.

---
## 📚 Başvuru: Bu defterde neyi nerede yaptık?

| İş | Nerede | Kod |
|---|---|---|
| Türleri 0/1 sütunlara açmak | Bölüm 0 | `str.get_dummies("|")` |
| Ortak tür sayısı | 1.1 | `(mini * satir).sum(axis=1)` |
| Kosinüs (elle) | 1.2 | `(a*b).sum() / (np.sqrt((a**2).sum()) * np.sqrt((b**2).sum()))` |
| Kosinüs (kütüphane), tek satır | Bölüm 2 | `cosine_similarity(X[[i]], X)[0]` |
| Beraberlik bozma | Bölüm 2 | `sort_values(["kosinus", "puan_sayisi"], ascending=False)` |
| Kullanıcı profili | Bölüm 3 | `turler.loc[begendi_idx].mean()` |
| Metinden TF-IDF matrisi | Bölüm 4–5 | `TfidfVectorizer().fit_transform(metin.fillna(""))` |
| Film başına etiket metni | Bölüm 4 | `groupby("movieId")["tag"].apply(" ".join)` |